In [ ]:
import subprocess
from pathlib import Path


def escape_subtitle_path(path: Path) -> str:
    p = path.resolve().as_posix()
    p = p.replace("\\", "/")
    p = p.replace("'", r"\'")
    p = p.replace(":", r"\:")
    return p


def burn_subtitles_into_video(
    video_path: str,
    subtitle_path: str,
    output_path: str = None,
    video_codec: str = "libx264",
    audio_codec: str = "aac",
    crf: int = 18,
    preset: str = "medium"
):
    ffmpeg_path = "/opt/homebrew/opt/ffmpeg-full/bin/ffmpeg"

    video_file = Path(video_path)
    subtitle_file = Path(subtitle_path)

    if not video_file.exists():
        raise FileNotFoundError(f"Video file not found: {video_path}")

    if not subtitle_file.exists():
        raise FileNotFoundError(f"Subtitle file not found: {subtitle_path}")

    if output_path is None:
        output_path = str(video_file.with_name(video_file.stem + "_subbed.mp4"))

    escaped_sub_path = escape_subtitle_path(subtitle_file)
    subtitle_filter = f"subtitles=filename='{escaped_sub_path}'"

    cmd = [
        ffmpeg_path,
        "-y",
        "-i", str(video_file),

        # only keep first video + optional audio
        "-map", "0:v:0",
        "-map", "0:a?",

        # do not copy any subtitle streams from the source
        "-sn",

        "-vf", subtitle_filter,
        "-c:v", video_codec,
        "-preset", preset,
        "-crf", str(crf),
        "-c:a", audio_codec,
        output_path
    ]

    print("Running command:")
    print(" ".join(cmd))

    result = subprocess.run(cmd, capture_output=True, text=True)

    print("\n--- FFmpeg stderr ---")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"FFmpeg failed with exit code {result.returncode}")

    print(f"\nDone. Output saved to: {output_path}")
    return output_path

In [ ]:
output_video = burn_subtitles_into_video(
    video_path="French_Classic.mp4",
    subtitle_path="French_Classic_english.srt",
    output_path="French_Classic_burned.mp4"
)